# Resolución de problemas de búsqueda

En este cuaderno representaremos los estados con estructuras sencillas y buscaremos una ruta hasta el objetivo. Lo haremos con dos ejemplos: las torres de Hanói y una ruta por Castellón.

## Torres de Hanói

### Representación del estado

Un estado es una tupla de tres torres. Cada torre es una tupla de discos ordenados de abajo hacia arriba. Por ejemplo, `((3, 2, 1), (), ())` indica que los tres discos están en la primera torre. Las tuplas se pueden guardar en un conjunto de visitados y en un diccionario de padres.

In [ ]:
def genera_estats(estat):
    estats = []

    # Prova a moure el disc superior de cada torre a una altra torre.
    for origen in range(3):
        if not estat[origen]:
            continue

        disc = estat[origen][-1]
        for desti in range(3):
            if origen == desti:
                continue

            # Només es pot posar un disc sobre una torre buida
            # o sobre un disc més gran.
            if not estat[desti] or disc < estat[desti][-1]:
                nou_estat = [list(torre) for torre in estat]
                nou_estat[desti].append(nou_estat[origen].pop())
                estats.append(tuple(tuple(torre) for torre in nou_estat))

    return estats

La función siguiente busca una solución con **BFS** (búsqueda en anchura). Como cada movimiento cuesta un paso, BFS encuentra una solución con el número mínimo de movimientos. El diccionario `pares` recuerda desde qué estado hemos llegado a cada uno.

In [ ]:
def bfs(inicial, es_objectiu, genera_estats):
    cua = [inicial]
    visitats = {inicial}
    pares = {inicial: None}

    while cua:
        estat = cua.pop(0)
        if es_objectiu(estat):
            return estat, pares

        for successor in genera_estats(estat):
            if successor not in visitats:
                visitats.add(successor)
                pares[successor] = estat
                cua.append(successor)

    return None, pares


def reconstrueix_ruta(pares, final):
    ruta = []
    while final is not None:
        ruta.append(final)
        final = pares[final]
    return list(reversed(ruta))

### Resolución

Empezamos con tres discos. La función objetivo comprueba si todos han llegado a la tercera torre.

In [ ]:
inicial = ((3, 2, 1), (), ())
final, pares = bfs(
    inicial,
    lambda estat: estat == ((), (), (3, 2, 1)),
    genera_estats
)

for pas, estat in enumerate(reconstrueix_ruta(pares, final)):
    print(f"Pas {pas}: {estat}")

## Ruta por Castellón

Ahora cada estado es el nombre de un lugar. El grafo guarda los lugares a los que podemos ir y el coste de cada trayecto. A diferencia de Hanói, aquí los trayectos pueden tener costes distintos.

In [ ]:
# Cada aresta indica el lloc veí i el cost del trajecte.
graf = {
    "UJI": [("Borrull", 4), ("Agustina", 6)],
    "Borrull": [("UJI", 4), ("Agustina", 2)],
    "Agustina": [("UJI", 6), ("Borrull", 2), ("Santa Clara", 3)],
    "Santa Clara": [("Agustina", 3), ("Major", 2), ("Ribalta", 8)],
    "Major": [("Santa Clara", 2), ("Ribalta", 5)],
    "Ribalta": [("Santa Clara", 8), ("Major", 5)],
}

### Búsqueda de coste uniforme y A*

La función mantiene el coste más barato conocido para cada lugar y sus padres para reconstruir la ruta.

- Con una heurística de ceros, prioriza el coste recorrido: es la **búsqueda de coste uniforme**.
- Con una estimación del coste que falta, prioriza `coste recorrido + estimación`: es **A\***.

Para que A* encuentre una ruta óptima, la estimación nunca debe superar el coste real que falta.

In [ ]:
import heapq


def cerca_cost_uniforme_o_a_estrella(inicial, final, heuristica):
    cua = [(heuristica[inicial], 0, inicial)]
    costos = {inicial: 0}
    pares = {inicial: None}

    while cua:
        prioritat, cost, estat = heapq.heappop(cua)

        # Ignora una entrada antiga si ja hem trobat un camí més barat.
        if cost != costos[estat]:
            continue

        if estat == final:
            return pares, cost

        for successor, cost_accio in graf[estat]:
            nou_cost = cost + cost_accio
            if successor not in costos or nou_cost < costos[successor]:
                costos[successor] = nou_cost
                pares[successor] = estat
                prioritat = nou_cost + heuristica[successor]
                heapq.heappush(cua, (prioritat, nou_cost, successor))

    return None, None


def mostra_ruta(pares, final):
    ruta = reconstrueix_ruta(pares, final)
    print(" → ".join(ruta))

### Búsqueda de coste uniforme

Con una heurística de ceros, encontramos la ruta de coste mínimo desde la UJI hasta el parque Ribalta.

In [ ]:
inicial = "UJI"
final = "Ribalta"
heuristica_zeros = {lloc: 0 for lloc in graf}

pares, cost = cerca_cost_uniforme_o_a_estrella(inicial, final, heuristica_zeros)
mostra_ruta(pares, final)
print("Coste total:", cost)

### A* con una estimación

Definimos una estimación del coste que falta para llegar a la Ribalta. En este mapa es admisible: no supera el coste real más barato desde cada lugar.

In [ ]:
heuristica = {
    "UJI": 10,
    "Borrull": 8,
    "Agustina": 6,
    "Santa Clara": 4,
    "Major": 3,
    "Ribalta": 0,
}

pares, cost = cerca_cost_uniforme_o_a_estrella(inicial, final, heuristica)
mostra_ruta(pares, final)
print("Coste total:", cost)

En los dos casos obtenemos una ruta de coste mínimo. La diferencia está en el orden de exploración: A* utiliza la estimación para orientar la búsqueda hacia el objetivo.